# BTL02 T0403 — EXP-12 LR Scheduler Ablation (Colab GPU)
Local-first training, atomic Drive sync, Auto-Skip MLflow FINISHED.

In [ ]:
import os, shutil, subprocess, sys, tempfile, time
from pathlib import Path
import torch
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật Colab GPU trước khi chạy T0403.')
from google.colab import drive, userdata
drive.mount('/content/drive')
BRANCH='exp/BTL02_T0403_ablation_lr_scheduler'
REPO_URL='https://github.com/Duy-DS/Deep_Learning_Group.git'
REPO=Path('/content/Deep_Learning_Group'); PROJECT=REPO/'BTL02'
DRIVE_ROOT=Path('/content/drive/MyDrive/BTL02/T0403_Scheduler')
DRIVE_MLRUNS=DRIVE_ROOT/'mlruns'; DRIVE_CKPT=DRIVE_ROOT/'checkpoints'; DRIVE_LOGS=DRIVE_ROOT/'logs'
LOCAL_MLRUNS=Path('/content/mlruns_t0403'); LOCAL_CKPT=Path('/content/checkpoints_t0403'); LOCAL_LOGS=Path('/content/logs_t0403')
for folder in (DRIVE_ROOT,DRIVE_MLRUNS,DRIVE_CKPT,DRIVE_LOGS,LOCAL_MLRUNS,LOCAL_CKPT,LOCAL_LOGS): folder.mkdir(parents=True,exist_ok=True)
(LOCAL_MLRUNS/'.trash').mkdir(exist_ok=True)
os.environ.update({'MLFLOW_TRACKING_URI':LOCAL_MLRUNS.as_uri(),'MLFLOW_ALLOW_FILE_STORE':'true'})
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
def sync_tree(source,destination,stable_age_seconds=0):
    source,destination=Path(source),Path(destination)
    if not source.exists(): return
    for root,dirs,files in os.walk(source):
        current=Path(root); dirs[:]=[d for d in dirs if d!='.trash']
        target_root=destination/current.relative_to(source); target_root.mkdir(parents=True,exist_ok=True)
        for name in files:
            item=current/name
            if item.is_symlink() or name.endswith(('.tmp','.part','.sync-tmp')): continue
            stat=item.stat()
            if stable_age_seconds and time.time()-stat.st_mtime<stable_age_seconds: continue
            target=target_root/name
            if target.exists() and target.stat().st_size==stat.st_size and target.stat().st_mtime_ns>=stat.st_mtime_ns: continue
            temp=target.with_name(target.name+'.sync-tmp'); shutil.copy2(item,temp)
            if temp.stat().st_size==stat.st_size: os.replace(temp,target)
            else: temp.unlink(missing_ok=True)
def run_git(args,env=None): subprocess.run(['git','-C',str(REPO),*args],check=True,env=env)
token=None
try: token=userdata.get('GITHUB_TOKEN')
except Exception: pass
git_env=os.environ.copy(); askpass=None
if token:
    fd,askpass=tempfile.mkstemp(prefix='t0403-askpass-')
    with os.fdopen(fd,'w') as f: f.write('#!/bin/sh\ncase \"$1\" in *Username*) echo x-access-token;; *) echo \"$T0403_GITHUB_TOKEN\";; esac\n')
    os.chmod(askpass,0o700); git_env.update({'GIT_ASKPASS':askpass,'GIT_TERMINAL_PROMPT':'0','T0403_GITHUB_TOKEN':token})
try:
    if REPO.exists(): run_git(['fetch','origin',BRANCH],git_env); run_git(['checkout',BRANCH],git_env); run_git(['reset','--hard','origin/'+BRANCH],git_env)
    else: subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO)],check=True,env=git_env)
finally:
    if askpass and os.path.exists(askpass): os.remove(askpass)
    git_env.pop('T0403_GITHUB_TOKEN',None); token=None
sync_tree(DRIVE_MLRUNS,LOCAL_MLRUNS); sync_tree(DRIVE_CKPT,LOCAL_CKPT)
(LOCAL_MLRUNS/'.trash').mkdir(exist_ok=True)
os.environ['MLFLOW_TRACKING_URI']=LOCAL_MLRUNS.as_uri()
PROJECT=REPO/'BTL02'; os.chdir(PROJECT)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()==BRANCH

In [ ]:
requirements=Path('requirements.txt').read_text(encoding='utf-8').splitlines()
packages=[line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=',1)[0].strip().lower() not in {'torch','torchvision'}]
subprocess.run([sys.executable,'-m','pip','install','-q',*packages],check=True)
from src.data.downloader import FER2013Downloader
from google.colab import userdata
raw=Path('data/raw/fer2013.csv'); raw.parent.mkdir(parents=True,exist_ok=True)
drive_csv=DRIVE_ROOT/'fer2013.csv'
if not raw.is_file() and drive_csv.is_file(): shutil.copy2(drive_csv,raw)
if not raw.is_file():
    token=userdata.get('KAGGLE_API_TOKEN')
    if not token: raise RuntimeError('Cần Secret KAGGLE_API_TOKEN hoặc fer2013.csv trên Drive.')
    os.environ['KAGGLE_API_TOKEN']=token; token=None
    try: subprocess.run([sys.executable,'-m','src.data.downloader'],check=True)
    finally: os.environ.pop('KAGGLE_API_TOKEN',None)
counts=FER2013Downloader.validate_csv(raw)
assert counts=={'Training':28709,'PublicTest':3589,'PrivateTest':3589},counts
if not drive_csv.exists(): shutil.copy2(raw,drive_csv)
outputs=Path('outputs'); outputs.mkdir(exist_ok=True)
for name,target in (('checkpoints',LOCAL_CKPT),('logs',LOCAL_LOGS)):
    link=outputs/name
    if link.exists() and not link.is_symlink():
        if any(p.name!='.gitkeep' for p in link.iterdir()): raise RuntimeError(f'Không ghi đè output có dữ liệu: {link}')
        for p in link.iterdir(): p.unlink()
        link.rmdir()
    if link.is_symlink() and link.resolve()!=target.resolve(): raise RuntimeError(f'Symlink không đúng: {link}')
    if not link.exists(): link.symlink_to(target,target_is_directory=True)
print('FER2013 splits:',counts)

In [ ]:
import pandas as pd, yaml
from mlflow.tracking import MlflowClient
from src.train.trainer import Trainer
from src.train.mlflow_logger import MLflowLogger
CONFIG_FILES=['exp12_sched_step_or_plateau.yaml','exp12_sched_cosine.yaml','exp12_sched_warmup_cosine.yaml']
configs=[]
for filename in CONFIG_FILES:
    cfg=yaml.safe_load((Path('experiments/ablation')/filename).read_text(encoding='utf-8'))
    cfg['run_name']=Trainer._format_run_name(exp_id=cfg['exp_id'],architecture=cfg['architecture'],optimizer=cfg['optimizer'],pretrained=cfg['pretrained'],image_size=cfg['image_size'],augmentation_mode=cfg['augmentation_mode'],loss_name=cfg['loss']['name'],scheduler_name=cfg['scheduler'])
    configs.append(cfg)
client=MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI']); exp=client.get_experiment_by_name('BTL02_FER2013_PyTorch'); finished=set()
if exp:
    for run in client.search_runs([exp.experiment_id],max_results=10000):
        if run.info.status=='FINISHED': finished.add(run.data.tags.get('mlflow.runName',''))
assert len(MLflowLogger.CSV_COLUMNS)==24
display(pd.DataFrame([{'config':f,'scheduler':c['scheduler'],'run_name':c['run_name'],'status':'FINISHED' if c['run_name'] in finished else 'PENDING'} for f,c in zip(CONFIG_FILES,configs)]))
CSV=LOCAL_LOGS/'t0403_ablation_scheduler_24col.csv'; DRIVE_CSV=DRIVE_LOGS/CSV.name
def export_sync():
    logger=MLflowLogger(); raw=LOCAL_LOGS/'t0403_all_runs_export.csv'; logger.export_summary_csv(str(raw))
    frame=pd.read_csv(raw,encoding='utf-8-sig',keep_default_na=False); assert frame.columns.tolist()==MLflowLogger.CSV_COLUMNS
    names={c['run_name'] for c in configs}; frame=frame[frame['Name'].isin(names)&frame['Status'].astype(str).str.upper().eq('FINISHED')]
    temp=CSV.with_name(CSV.name+'.sync-tmp'); frame.to_csv(temp,index=False,encoding='utf-8-sig'); os.replace(temp,CSV)
    drive_tmp=DRIVE_CSV.with_name(DRIVE_CSV.name+'.sync-tmp'); shutil.copy2(CSV,drive_tmp); os.replace(drive_tmp,DRIVE_CSV)
    sync_tree(LOCAL_MLRUNS,DRIVE_MLRUNS,stable_age_seconds=3); sync_tree(LOCAL_CKPT,DRIVE_CKPT); sync_tree(LOCAL_LOGS,DRIVE_LOGS); raw.unlink(missing_ok=True)
    return frame
failures=[]
for filename,cfg in zip(CONFIG_FILES,configs):
    if cfg['run_name'] in finished: print('[AUTO-SKIP FINISHED]',cfg['run_name']); export_sync(); continue
    result=subprocess.run([sys.executable,'-u','-m','src.train.trainer','--config',str(Path('experiments/ablation')/filename)],cwd=PROJECT)
    if result.returncode: failures.append((filename,result.returncode))
    export_sync()
final=export_sync(); assert final.columns.tolist()==MLflowLogger.CSV_COLUMNS
print('Finished rows:',len(final),'/3; failures:',failures,'CSV:',DRIVE_CSV)
display(final[['Name','Run ID','Status','metrics.best_epoch','metrics.train_loss','metrics.train_acc','metrics.best_val_acc','metrics.private_test_acc','metrics.macro_f1']])